<h1><b>Order By</b></h1>
Order your results to focus on the most important data for your use case.

# <h2 id="Introduction">Introduction</h2>

<p>So far, you've learned how to use several SQL clauses.  For instance, you know how to use <strong>SELECT</strong> to pull specific columns from a table, along with <strong>WHERE</strong> to pull rows that meet specified criteria.  You also know how to use aggregate functions like <strong>COUNT()</strong>, along with <strong>GROUP BY</strong> to treat multiple rows as a single group.</p>

<p>Now you'll learn how to change the order of your results using the <strong>ORDER BY</strong> clause, and you'll explore a popular use case by applying ordering to dates.  To illustrate what you'll learn in this tutorial, we'll work with a slightly modified version of our familiar <code>pets</code> table.</p>

<p><img src="https://storage.googleapis.com/kaggle-media/learn/images/b99zTLv.png"></p>

# <h2 id="ORDER-BY">ORDER BY</h2>

<p><strong>ORDER BY</strong> is usually the last clause in your query, and it sorts the results returned by the rest of your query.</p>
<p>Notice that the rows are not ordered by the <code>ID</code> column.  We can quickly remedy this with the query below.</p>

<p><img src="https://storage.googleapis.com/kaggle-media/learn/images/6o9LuTA.png"></p>

<p>The <strong>ORDER BY</strong> clause also works for columns containing text, where the results show up in alphabetical order.</p>
<p><img src="https://storage.googleapis.com/kaggle-media/learn/images/ooxuzw3.png"></p>

<p>You can reverse the order using the <strong>DESC</strong> argument (short for 'descending').  The next query sorts the table by the <code>Animal</code> column, where the values that are last in alphabetic order are returned first.</p>

<p><img src="https://storage.googleapis.com/kaggle-media/learn/images/IElLJrR.png"></p>

# <h2 id="Dates">Dates</h2>

<p>Next, we'll talk about dates, because they come up very frequently in real-world databases. There are two ways that dates can be stored in BigQuery: as a <strong>DATE</strong> or as a <strong>DATETIME</strong>.</p>

<p>The <strong>DATE</strong> format has the year first, then the month, and then the day. It looks like this:</p>

<pre><code>YYYY-[M]M-[D]D</code></pre>
<ul>
<li><code>YYYY</code>: Four-digit year</li>
<li><code>[M]M</code>: One or two digit month</li>
<li><code>[D]D</code>: One or two digit day</li>
</ul>

<p>So <code>2019-01-10</code> is interpreted as January 10, 2019.</p>

<p>The <strong>DATETIME</strong> format is like the date format ... but with time added at the end.</p>

# <h2 id="EXTRACT">EXTRACT</h2>

<p>Often you'll want to look at part of a date, like the year or the day. You can do this with <strong>EXTRACT</strong>.  We'll illustrate this with a slightly different table, called <code>pets_with_date</code>.</p>

<p><img src="https://storage.googleapis.com/kaggle-media/learn/images/vhvHIh0.png"></p>

<p>The query below returns two columns, where column <code>Day</code> contains the day corresponding to each entry the <code>Date</code> column from the <code>pets_with_date</code> table:</p>

<p><img src="https://storage.googleapis.com/kaggle-media/learn/images/PhoWBO0.png"></p>

<p>SQL is very smart about dates, and we can ask for information beyond just extracting part of the cell. For example, this query returns one column with just the week in the year (between 1 and 53) for each date in the <code>Date</code> column:</p>

<p><img src="https://storage.googleapis.com/kaggle-media/learn/images/A5hqGxY.png"></p>

<p>You can find all the functions you can use with dates in BigQuery in <a href="https://cloud.google.com/bigquery/docs/reference/legacy-sql#datetimefunctions">this documentation</a> under "Date and time functions".</p>

# <h2>Example: Which day of the week has the most fatal motor accidents?</h2>

<p>Let's use the US Traffic Fatality Records database, which contains information on traffic accidents in the US where at least one person died.</p>

<p>We'll investigate the <code>accident_2015</code> table. Here is a view of the first few rows.</p>

In [2]:
from google.cloud import bigquery

# Project ID
project_id = "burnished-road-363918"

# Create a "Client" object
client = bigquery.Client(project=project_id)

# Authenticate
from google.colab import auth
auth.authenticate_user()

# Construct a reference to the "nhtsa_traffic_fatalities" dataset
dataset_ref = client.dataset("nhtsa_traffic_fatalities", project="bigquery-public-data")

# API request - fetch the dataset
dataset = client.get_dataset(dataset_ref)

# Construct a reference to the "accident_2015" table
table_ref = dataset_ref.table("accident_2015")

# API request - fetch the table
table = client.get_table(table_ref)

# Preview the first five lines of the "accident_2015" table
client.list_rows(table, max_results=5).to_dataframe()

,state_number,state_name,consecutive_number,number_of_vehicle_forms_submitted_all,number_of_motor_vehicles_in_transport_mvit,number_of_parked_working_vehicles,number_of_forms_submitted_for_persons_not_in_motor_vehicles,number_of_persons_not_in_motor_vehicles_in_transport_mvit,number_of_persons_in_motor_vehicles_in_transport_mvit,number_of_forms_submitted_for_persons_in_motor_vehicles,...,minute_of_ems_arrival_at_hospital,related_factors_crash_level_1,related_factors_crash_level_1_name,related_factors_crash_level_2,related_factors_crash_level_2_name,related_factors_crash_level_3,related_factors_crash_level_3_name,number_of_fatalities,number_of_drunk_drivers,timestamp_of_crash
0,30,Montana,300019,5,5,0,0,0,7,7,...,45,0,None,0,None,0,None,1,0,2015-03-28 14:58:00+00:00
1,39,Ohio,390099,7,7,0,0,0,15,15,...,24,27,Backup Due to Prior Crash,0,None,0,None,1,0,2015-02-14 11:19:00+00:00
2,49,Utah,490123,16,16,0,0,0,28,28,...,99,0,None,0,None,0,None,1,0,2015-04-14 12:24:00+00:00
3,48,Texas,481184,6,5,1,0,5,5,10,...,99,0,None,0,None,0,None,1,0,2015-05-27 16:40:00+00:00
4,41,Oregon,410333,11,11,0,0,0,14,14,...,99,0,None,0,None,0,None,1,0,2015-11-17 18:17:00+00:00


<p>Let's use the table to determine how the number of accidents varies with the day of the week. Since:</p>

<ul>
<li>the <code>consecutive_number</code> column contains a unique ID for each accident, and</li>
<li>the <code>timestamp_of_crash</code> column contains the date of the accident in DATETIME format,</li>
</ul>

<p>we can:</p>

<ul>
<li><strong>EXTRACT</strong> the day of the week (as <code>day_of_week</code> in the query below) from the <code>timestamp_of_crash</code> column, and</li>
<li><strong>GROUP BY</strong> the day of the week, before we <strong>COUNT</strong> the <code>consecutive_number</code> column to determine the number of accidents for each day of the week.</li>
</ul>

<p>Then we sort the table with an <strong>ORDER BY</strong> clause, so the days with the most accidents are returned first.</p>

In [3]:
# Query to find out the number of accidents for each day of the week
query = """
        SELECT COUNT(consecutive_number) AS num_accidents,
               EXTRACT(DAYOFWEEK FROM timestamp_of_crash) AS day_of_week
        FROM `bigquery-public-data.nhtsa_traffic_fatalities.accident_2015`
        GROUP BY day_of_week
        ORDER BY num_accidents DESC
        """

<p>As usual, we run it as follows:</p>

In [4]:
# Set up the query (cancel the query if it would use too much of
# your quota, with the limit set to 1 GB)
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**9)
query_job = client.query(query, job_config=safe_config)

# API request - run the query, and convert the results to a pandas DataFrame
accidents_by_day = query_job.to_dataframe()

# Print the DataFrame
accidents_by_day

,num_accidents,day_of_week
0,5659,7
1,5298,1
2,4916,6
3,4460,5
4,4182,4
5,4038,2
6,3985,3


<p>Notice that the data is sorted by the <code>num_accidents</code> column, where the days with more traffic accidents appear first.</p>
<p>To map the numbers returned for the <code>day_of_week</code> column to the actual day, you might consult <a href="https://cloud.google.com/bigquery/docs/reference/legacy-sql#dayofweek">the BigQuery documentation</a> on the DAYOFWEEK function. It says that it returns "an integer between 1 (Sunday) and 7 (Saturday), inclusively". So, in 2015, most fatal motor accidents in the US occured on Sunday and Saturday, while the fewest happened on Tuesday.</p>